# Package 1: 基于大语言模型引导的真实噪声图像数据准备与语义标注框架

## 📋 概述

本教程包聚焦于图像去噪任务的第一步：构建高质量、语义丰富的含噪图像数据集。我们将利用真实世界拍摄的低光/高ISO图像作为基础，并引入GPT类大语言模型（LLM）自动生成噪声类型（如高斯、泊松、椒盐）及其强度标签，从而实现高效、低成本的噪声语义标注。该数据集不仅包含像素级噪声样本，还附带文本描述（如“中等高斯噪声，源于夜间手持拍摄”），为后续LLM引导的扩散去噪模型提供关键监督信号。此步骤是整个研究目标的基础——没有真实且语义对齐的数据，再先进的模型也无法学习到鲁棒的去噪能力。


## 📂 项目结构

```
package-01-llm-noise-annotation/
├── README.md
├── requirements.txt
├── src/
│   ├── main.py
│   ├── data_collector.py
│   ├── llm_annotator.py
│   ├── noise_simulator.py
│   └── augmenter.py
├── configs/
│   └── config.yaml
├── data/
│   ├── raw_images/
│   │   └── *.jpg
│   ├── annotated_dataset.json
│   └── augmented_samples/
└── docs/
    └── usage.md
```


## 💡 理论基础

同学们，今天我们来探讨一个看似简单却极其关键的问题：**如何为图像去噪任务准备既真实又带有语义标签的数据？** 你可能会想：“不就是加点噪声吗？”但现实远比这复杂。传统方法常在干净图像上人工添加合成噪声（如固定方差的高斯噪声），但这与真实相机传感器在低光下产生的复杂噪声分布相去甚远 [Ho, 2020]。真实噪声往往是信号相关的（signal-dependent），例如泊松噪声的强度随像素亮度变化，而读出噪声则接近高斯分布——这种混合特性使得单一噪声模型难以准确刻画。

为了解决这一问题，我们的第一步不是建模，而是**数据采集与语义化标注**。我们主张从真实场景出发：收集大量由普通用户或专业摄影师在弱光、高ISO条件下拍摄的图像。这些图像天然包含复杂的噪声模式，但缺乏明确的“噪声类型”和“强度”标签。手动标注不仅成本高昂，而且人类也难以精确判断噪声的数学分布。这时，大语言模型（LLM）就派上了用场。

我们提出一种**LLM辅助的噪声语义标注机制**。给定一张含噪图像及其元数据（如ISO值、快门速度、相机型号），我们将这些信息转化为自然语言提示（prompt），例如：“这张照片使用Canon EOS R5在ISO 6400、1/30秒快门下拍摄，画面整体偏暗，细节模糊，有明显彩色噪点。” 然后，我们将此提示输入GPT-4等LLM，要求其输出结构化标签：`{"noise_type": ["gaussian", "poisson"], "intensity": "medium", "source": "low_light_handheld"}`。这种方法借鉴了InstructBLIP中指令调优的思想 [Zhou, 2022]，将LLM视为一个强大的语义推理引擎，而非生成器。

从理论上讲，这一过程可形式化为一个**条件概率映射**：
$$P(\mathcal{L} \mid \mathcal{I}, \mathcal{M})$$
其中 $\mathcal{I}$ 是输入图像，$\mathcal{M}$ 是元数据（metadata），$\mathcal{L}$ 是输出的噪声语义标签。LLM通过其预训练的语言理解能力，将视觉现象（通过文本描述）映射到技术术语，实现了跨模态的语义对齐。值得注意的是，我们并不要求LLM“看到”图像，而是依赖人类撰写的图像描述——这避免了直接多模态输入的复杂性，同时保持了高准确性。

为什么选择这种方式？首先，它**大幅降低标注成本**。相比雇佣专家逐张分析噪声分布，LLM可在秒级内完成推理。其次，它**提升标签的语义丰富度**。传统方法只能标注“噪声标准差=25”，而LLM可输出“中等强度高斯-泊松混合噪声，源于夜间手持拍摄”，这对后续的语义引导去噪至关重要。正如ELLA工作所示，细粒度的文本条件能显著提升扩散模型的生成一致性 [Chen, 2024]。

当然，这一方法也有权衡。LLM可能产生幻觉（hallucination），例如将压缩伪影误判为椒盐噪声。为此，我们设计了**置信度过滤机制**：仅保留LLM输出概率高于阈值的标签，并辅以少量人工校验。此外，我们结合**数据增强策略**（如旋转、裁剪、色彩抖动）扩充样本，但严格避免改变噪声统计特性——例如不使用高斯模糊，以免掩盖原始噪声结构。

在数学上，我们的噪声模拟模块也需谨慎设计。对于真实噪声近似，我们采用**异方差高斯-泊松混合模型**：
$$y = x + \sqrt{x}\cdot \epsilon_p + \sigma_r \cdot \epsilon_g$$
其中 $x$ 是干净图像，$y$ 是观测图像，$\epsilon_p \sim \mathcal{N}(0,1)$ 表示泊松噪声（与信号相关），$\epsilon_g \sim \mathcal{N}(0,1)$ 表示读出高斯噪声，$\sigma_r$ 为其标准差。该模型已被证明能较好拟合CMOS传感器噪声 [Hasinoff, 2010]。但在本步骤中，我们**优先使用真实图像**，仅在样本不足时用此模型进行可控增强。

最后，这一数据准备流程直接服务于后续的LLM引导扩散去噪框架。如ControlNet所示，条件控制的质量决定了生成结果的保真度 [Zhang, 2023]。如果我们输入的文本提示是“去除中等高斯噪声，保留纹理细节”，那么模型必须在训练阶段见过大量对应的真实样本。因此，本步骤不仅是数据工程，更是**语义对齐的奠基工作**。

总结一下：真实噪声数据 + LLM语义标注 + 谨慎增强 = 高质量、语义丰富的训练集。这为后续构建PSNR≥35dB、SSIM≥0.92的去噪模型打下坚实基础。记住，**垃圾进，垃圾出（Garbage in, garbage out）**——再强大的模型也需要好数据！


---

## 📖 核心概念详解

在开始实现之前，请先理解以下核心概念。这些概念是理解本包实现的关键前提。


### 真实世界噪声建模（Real-World Noise Modeling）

同学们，让我们从一个日常场景开始：你在夜晚用手机拍照，发现照片有很多彩色小点，细节模糊不清。这些就是“噪声”。但你有没有想过，这些噪声到底是什么？它们遵循什么规律？

在计算机视觉中，“噪声”指的是图像中非真实场景内容的随机干扰。早期研究假设噪声是简单的高斯分布——即每个像素独立地加上一个均值为0、标准差固定的随机数。数学表达为：$y = x + n$，其中 $n \sim \mathcal{N}(0, \sigma^2)$。这种模型计算简单，但**严重脱离现实**。

真实相机传感器产生的噪声要复杂得多。主要有两类：**光子散粒噪声（Photon Shot Noise）** 和 **读出噪声（Read Noise）**。前者源于光子到达传感器的随机性，服从泊松分布；后者源于电子电路的热扰动，接近高斯分布。更关键的是，泊松噪声的强度与像素亮度成正比——越亮的地方，噪声越大！这称为“信号相关噪声”（signal-dependent noise）。

因此，现代真实噪声模型采用**高斯-泊松混合模型**：
$$y = x + \sqrt{x} \cdot \epsilon_p + \sigma_r \cdot \epsilon_g$$
这里，$x$ 是理想干净图像（单位为光子数），$\epsilon_p$ 和 $\epsilon_g$ 都是标准正态分布随机变量。$\sqrt{x}$ 项体现了泊松噪声的方差等于均值的特性（因为泊松分布的方差=均值），而 $\sigma_r$ 控制读出噪声的强度。这个公式看似简单，却能很好地拟合大多数数码相机的噪声行为 [Hasinoff, 2010]。

为什么这对去噪如此重要？因为如果你用纯高斯模型训练去噪网络，它会假设所有区域的噪声强度相同。但在真实图像中，暗区主要是读出噪声（较均匀），亮区则叠加了强泊松噪声（更杂乱）。如果模型不知道这一点，它要么过度平滑亮区细节，要么无法有效抑制暗区噪声。

举个生活中的例子：想象你在雨中听音乐。雨滴声（类似泊松噪声）在鼓点强时更响，在安静段落时较弱；而耳机本身的电流声（类似读出噪声）始终存在。如果你只根据安静时的雨声来设计降噪算法，那么在高潮部分就会失效。

另一个例子是医学影像。X光图像的噪声也与辐射剂量相关——剂量越高，图像越亮，但光子噪声也越大。忽略这种关系会导致误诊。

在本教程包中，我们不直接依赖合成噪声模型，而是**优先采集真实含噪图像**。因为即使最先进的混合模型也无法完全捕捉传感器非线性、色彩滤镜阵列插值、ISP处理等复杂因素。真实数据才是黄金标准。

然而，真实数据稀缺且标注困难。这时，我们可以用上述混合模型进行**可控增强**：在已知干净图像上，按特定 $\sigma_r$ 和光照条件添加噪声，生成“半真实”样本。但必须注意：增强后的图像不能用于最终评估，只能辅助训练。

总之，理解真实噪声的本质，是构建有效去噪系统的第一步。它告诉我们：**噪声不是敌人，而是携带场景信息的信号**。我们的目标不是盲目抹除，而是在理解其来源的基础上智能抑制。

这一理念也呼应了扩散模型的核心思想：去噪是一个逐步还原的过程，每一步都需考虑当前噪声的统计特性 [Ho, 2020]。而LLM引导则进一步将这种理解提升到语义层面——不仅知道“噪声多大”，还知道“为什么有噪声”。

**为什么重要**: 真实噪声建模是本数据准备步骤的理论基石。只有准确理解真实噪声的物理来源和数学特性，才能合理设计数据采集策略、评估LLM标注的合理性，并在必要时进行可信的数据增强。若忽略此概念，后续模型将在合成噪声上过拟合，无法泛化到真实场景。

**相关概念**: 信号相关噪声（Signal-Dependent Noise）, 泊松分布（Poisson Distribution）, 传感器噪声（Sensor Noise）, 图像信号处理器（ISP）

**示例与类比**:

- 夜间手机拍照出现的彩色噪点——主要由高ISO下的读出噪声和光子散粒噪声混合造成
- 天文摄影中的长曝光图像——暗电流噪声随曝光时间累积，表现为固定模式噪声叠加随机噪声
- 老式胶片照片的颗粒感——虽然非电子噪声，但同样具有信号相关性：高光区域颗粒更明显



### 大语言模型辅助语义标注（LLM-Assisted Semantic Annotation）

现在，让我们思考一个问题：如何给一张含噪图像打上“有意义”的标签？传统做法可能是测量噪声标准差，得到一个数字如“σ=25”。但这对人类或高级AI来说都不够直观。我们需要的是像“这张图有中等强度的彩色高斯噪声，源于低光手持拍摄”这样的**自然语言描述**。

这就是大语言模型（LLM）大显身手的地方。LLM（如GPT-4）经过海量文本训练，掌握了丰富的领域知识，包括摄影、图像处理、传感器原理等。我们可以把它当作一个“专家顾问”，帮我们解读图像现象背后的成因。

具体怎么做？我们不直接给LLM看图像（因为纯文本LLM无法处理像素），而是提供**人类撰写的图像描述**。例如：“使用iPhone 14 Pro在夜晚室内拍摄，ISO 2500，快门1/15秒，画面右侧有明显红绿噪点，皮肤纹理模糊。” 这个描述包含了关键线索：高ISO、慢快门（暗示手持抖动）、彩色噪点（暗示传感器热噪声）。

然后，我们设计一个结构化提示（prompt）：
```
你是一位图像处理专家。请根据以下描述，判断图像中的噪声类型、强度和可能来源。
描述："[上述文本]"
请以JSON格式输出，包含字段：noise_type（列表，选项：gaussian/poisson/salt_pepper/none），intensity（low/medium/high），source（如low_light, high_iso, motion_blur等）。
```

LLM会基于其内部知识推理出答案。例如，它知道高ISO通常导致读出噪声（高斯型），而极低光下光子稀缺会产生泊松噪声。这种推理能力源于其在训练中接触过大量技术文档和论坛讨论 [Brown, 2020]。

从数学角度看，这相当于学习一个映射函数 $f: \mathcal{D} \rightarrow \mathcal{L}$，其中 $\mathcal{D}$ 是自然语言描述空间，$\mathcal{L}$ 是结构化标签空间。LLM通过其Transformer架构中的自注意力机制，捕捉描述中的关键词（如“ISO 2500”、“彩色噪点”）并关联到噪声知识库。

为什么不用计算机视觉模型直接分析图像？因为：(1) 训练一个噪声分类器需要大量已标注的真实噪声图像，而这正是我们试图解决的鸡生蛋问题；(2) LLM的零样本（zero-shot）能力使其无需微调即可处理新场景。

举个类比例子：就像医生通过病人描述“头痛、发烧、喉咙痛”来诊断是流感还是新冠，LLM通过“高ISO、彩色噪点”推断噪声类型。它不依赖仪器（图像像素），而是依赖症状描述（文本）。

另一个例子是汽车维修。老师傅听到引擎异响，就能说出“可能是正时链条松动”。LLM就像这位老师傅，只不过它的“经验”来自互联网文本。

当然，LLM可能出错。比如将JPEG压缩块效应误认为椒盐噪声。因此，我们引入**置信度评分**：LLM在输出时可附带概率（如通过logits计算），我们只保留高置信度结果。此外，可设计**多轮验证**：让LLM自我质疑“这个判断合理吗？”，提升可靠性。

在本项目中，这种标注方式有三大优势：(1) **低成本**：自动化生成标签；(2) **高语义**：标签包含上下文信息；(3) **可扩展**：轻松适应新噪声类型（如未来新型传感器噪声），只需更新提示词。

最后，这种LLM辅助标注的思想，与InstructBLIP中的指令调优一脉相承 [Zhou, 2022]——都是利用LLM将任务转化为自然语言理解问题，从而释放其强大泛化能力。

**为什么重要**: LLM辅助语义标注是本步骤的核心创新。它解决了真实噪声数据缺乏语义标签的关键瓶颈，为后续LLM引导的扩散去噪提供高质量条件信号。没有这种细粒度、语义丰富的标注，文本提示将无法有效指导去噪过程。

**相关概念**: 零样本学习（Zero-Shot Learning）, 提示工程（Prompt Engineering）, 结构化输出（Structured Output Generation）, 置信度校准（Confidence Calibration）

**示例与类比**:

- 输入描述：“Sony A7III，ISO 12800，夜景人像，面部有明显彩色噪点” → 输出：{noise_type: ["gaussian"], intensity: "high", source: "high_iso_low_light"}
- 输入描述：“监控摄像头白天录像，画面有随机黑白点” → 输出：{noise_type: ["salt_pepper"], intensity: "low", source: "sensor_defect"}
- 输入描述：“扫描的老照片，整体颗粒感强但无彩色噪点” → 输出：{noise_type: ["gaussian"], intensity: "medium", source: "film_grain"}



## 🔧 实现步骤


### 1.1 数据收集器

**文件**: `src/data_collector.py`

**目的**: 从本地目录或网络来源收集真实世界含噪声图像，并提取其EXIF元数据（如ISO、快门速度、光圈值），为后续LLM语义标注提供上下文信息。

#### 详细说明

同学们，欢迎来到我们构建LLM引导去噪系统的第一步！在上一节的理论铺垫中，我们强调了**真实噪声数据的重要性**——合成噪声虽然可控，但无法反映相机传感器在低光、高ISO等极端条件下的复杂噪声行为。因此，我们的第一步不是写模型，而是**采集真实含噪图像及其拍摄上下文**。

这个`DataCollector`组件就是整个数据流水线的起点。它的核心任务是：遍历指定文件夹（比如你手机或相机导出的照片），读取每张JPEG/TIFF图像，并自动解析其EXIF元数据。这些元数据（如ISO=3200、快门=1/30s）是理解噪声来源的关键线索。例如，高ISO通常意味着更强的读出噪声，而慢速快门可能导致运动模糊与热噪声混合。

为什么我们要专门写一个收集器？因为直接使用原始图像而不记录其拍摄条件，就等于丢掉了噪声的“病因”。后续LLM需要这些信息来生成合理的噪声标签（比如“高ISO导致的彩色散粒噪声”）。如果我们跳过这一步，后续的语义标注就会变成无源之水。

在实现上，我们使用Python的`Pillow`库读取图像，用`piexif`解析EXIF。我们会过滤掉没有EXIF的图像（比如截图或经过多次压缩的网络图片），因为它们缺乏关键上下文。同时，我们只保留RGB格式的图像，避免CMYK等专业格式带来的兼容性问题。

数据流非常清晰：输入是一个包含原始图像的目录路径；输出是一个结构化的字典列表，每个元素包含：图像路径、加载后的PIL图像对象、以及解析出的EXIF字典（仅保留我们关心的字段：ISO、快门、光圈、相机型号、拍摄时间）。这些数据将被序列化后传递给下一步的`LLMAnnotator`。

设计上我们做了几个关键选择：第一，不立即加载所有图像到内存，而是按需读取路径，避免内存爆炸；第二，对EXIF字段做标准化处理（比如将快门'1/60'转为浮点数0.0167），方便后续LLM理解；第三，加入严格的错误处理——如果某张图损坏或EXIF损坏，我们记录警告但继续处理其他图像，保证流程鲁棒性。

举个例子：假设你有一张`night_photo.jpg`，EXIF显示ISO=6400，快门=1/15s。我们的收集器会输出：
```python
{
  'image_path': 'data/raw_images/night_photo.jpg',
  'image': <PIL.Image object>,
  'exif': {'iso': 6400, 'shutter_speed': 0.0667, 'aperture': 2.8, ...}
}
```
这个结构将直接喂给LLM提示工程模块。

边缘情况我们也考虑到了：有些手机照片的EXIF可能把ISO藏在MakerNote里，我们暂时忽略这些非标准字段；对于视频帧或GIF，我们只取第一帧并警告用户。我们的原则是：宁可少收，不可错收。

最后，这个组件与整体系统的衔接点在于：它为`llm_annotator.py`提供了带上下文的原始样本。没有它，LLM就只能“盲猜”噪声类型，准确率必然大打折扣。记住：**高质量的输入决定高质量的标注**。


In [ ]:
import osimport loggingfrom typing import List, Dict, Optional, Tuplefrom PIL import Imageimport piexif# 配置日志logging.basicConfig(level=logging.INFO)logger = logging.getLogger(__name__)class DataCollector:    """    数据收集器：从指定目录收集真实世界含噪声图像，并提取关键EXIF元数据。        功能：    - 遍历目录，筛选支持的图像格式（.jpg, .jpeg, .png, .tiff）    - 读取每张图像的EXIF信息，提取ISO、快门速度、光圈等关键字段    - 返回结构化数据列表，供后续LLM标注使用        参数:        data_dir (str): 原始图像所在目录路径            返回:        List[Dict]: 每个元素包含'image_path', 'image' (PIL.Image), 'exif' (dict)            示例:        collector = DataCollector("data/raw_images")        samples = collector.collect()        # samples[0]['exif']['iso'] -> 3200    """        def __init__(self, data_dir: str):        """        初始化数据收集器。                Args:            data_dir (str): 原始图像目录路径        """        if not os.path.isdir(data_dir):            raise ValueError(f"指定的数据目录不存在: {data_dir}")        self.data_dir = data_dir        # 支持的图像扩展名（小写）        self.supported_extensions = {'.jpg', '.jpeg', '.png', '.tiff', '.tif'}        def _is_valid_image(self, filepath: str) -> bool:        """        检查文件是否为支持的图像格式。                Args:            filepath (str): 文件完整路径                    Returns:            bool: 是否为有效图像        """        _, ext = os.path.splitext(filepath)        return ext.lower() in self.supported_extensions        def _parse_exif(self, image_path: str) -> Optional[Dict[str, any]]:        """        从图像中解析关键EXIF元数据。                提取字段：        - iso: 感光度        - shutter_speed: 快门速度（秒，浮点数）        - aperture: 光圈值        - camera_model: 相机型号        - datetime: 拍摄时间                Args:            image_path (str): 图像路径                    Returns:            Optional[Dict]: 解析后的EXIF字典，若失败则返回None        """        try:            exif_dict = piexif.load(image_path)            exif_info = {}                        # 尝试从Exif IFD中获取ISO            if "Exif" in exif_dict:                exif_ifd = exif_dict["Exif"]                # ISO Speed Ratings (tag 34855)                if 34855 in exif_ifd:                    exif_info["iso"] = exif_ifd[34855]                                # ApertureValue (tag 37378) - 注意这是APEX值，需转换                if 37378 in exif_ifd:                    # 简化处理：直接取值，实际应用中应转换为f-number                    exif_info["aperture"] = exif_ifd[37378][0] / exif_ifd[37378][1] if isinstance(exif_ifd[37378], tuple) else exif_ifd[37378]                        # 从主IFD获取快门速度和相机型号            if "0th" in exif_dict:                zeroth_ifd = exif_dict["0th"]                # Shutter Speed Value (tag 37377) - APEX值，需转换                if 37377 in zeroth_ifd:                    ss_val = zeroth_ifd[37377]                    if isinstance(ss_val, tuple):                        ss_val = ss_val[0] / ss_val[1]                    # 转换为实际快门速度（秒）: 1 / (2^ss_val)                    try:                        exif_info["shutter_speed"] = 1.0 / (2 ** ss_val)                    except (OverflowError, ZeroDivisionError):                        logger.warning(f"快门速度计算异常: {ss_val} in {image_path}")                                # Camera Model (tag 272)                if 272 in zeroth_ifd:                    model = zeroth_ifd[272]                    if isinstance(model, bytes):                        model = model.decode('utf-8', errors='ignore').strip('\x00')                    exif_info["camera_model"] = model                                # DateTime (tag 306)                if 306 in zeroth_ifd:                    dt = zeroth_ifd[306]                    if isinstance(dt, bytes):                        dt = dt.decode('ascii', errors='ignore')                    exif_info["datetime"] = dt                        return exif_info if exif_info else None                    except Exception as e:            logger.warning(f"解析EXIF失败 {image_path}: {str(e)}")            return None        def collect(self) -> List[Dict[str, any]]:        """        执行数据收集主流程。                步骤：        1. 遍历data_dir下所有文件        2. 过滤出支持的图像格式        3. 尝试加载图像并解析EXIF        4. 仅保留有有效EXIF的样本                Returns:            List[Dict]: 结构化样本列表        """        collected_samples = []                # 遍历目录        for root, _, files in os.walk(self.data_dir):            for file in files:                filepath = os.path.join(root, file)                                # 检查是否为有效图像                if not self._is_valid_image(filepath):                    continue                                try:                    # 尝试打开图像（验证是否损坏）                    with Image.open(filepath) as img:                        # 转换为RGB（处理RGBA/P等模式）                        if img.mode != 'RGB':                            img = img.convert('RGB')                        # 注意：这里不立即加载到内存，只保留路径                        # 实际图像加载推迟到需要时（节省内存）                        pass                                        # 解析EXIF                    exif_data = self._parse_exif(filepath)                    if exif_data is None:                        logger.info(f"跳过无EXIF图像: {filepath}")                        continue                                        # 构建样本字典                    sample = {                        'image_path': filepath,                        'exif': exif_data                    }                    collected_samples.append(sample)                                    except Exception as e:                    logger.warning(f"处理图像失败 {filepath}: {str(e)}")                    continue                logger.info(f"成功收集 {len(collected_samples)} 张带EXIF的图像")        return collected_samples

#### 重要提示

- EXIF解析的健壮性至关重要：不同相机厂商的EXIF结构差异很大，我们只提取通用字段，避免因特定厂商格式导致崩溃。实际部署时可扩展支持更多字段。
- 内存管理策略：我们不在收集阶段加载完整图像到内存，只保存路径。这样即使处理上万张图也不会内存溢出，图像加载推迟到数据增强或训练阶段按需进行。
- 格式兼容性：虽然PNG理论上可含EXIF，但实践中很少见。我们仍支持它，但预期大部分有效样本来自JPEG/TIFF。对于无EXIF的图像，直接跳过而非报错，保证流程继续。
- 快门速度转换的数学细节：EXIF中的快门速度存储为APEX值（ShutterSpeedValue），需通过公式 1/(2^SSV) 转换为秒。我们做了异常捕获防止指数运算溢出。


### 1.2 LLM噪声标注器

**文件**: `src/llm_annotator.py`

**目的**: 利用大语言模型（如GPT）根据图像EXIF元数据生成噪声类型（高斯、泊松、椒盐）和强度（低、中、高）的语义标签，并输出自然语言描述，用于构建带语义监督的训练数据集。

#### 详细说明

同学们，现在我们有了带EXIF的真实图像（来自上一步`DataCollector`的输出），但这些数据还缺少最关键的要素：**噪声的语义标签**。人类专家很难精确判断一张图像是高斯噪声还是泊松噪声主导，更别说量化强度了。这时，大语言模型（LLM）就成为我们的‘噪声诊断专家’。

`LLMAnnotator`的核心思想是：将EXIF元数据转化为自然语言提示（prompt），让LLM基于其海量知识推断最可能的噪声类型和强度。例如，给定ISO=6400、快门=1/15s，LLM可能输出：‘高斯噪声（高强度），源于高ISO读出噪声’。这种语义标签不仅告诉模型‘是什么噪声’，还解释了‘为什么’，为后续扩散模型的条件去噪提供丰富上下文。

为什么不用传统方法自动分类噪声？因为真实噪声往往是混合的（高斯+泊松+条纹噪声），且与场景内容耦合。LLM的优势在于能结合拍摄条件、相机型号甚至常识（如‘夜间手持拍摄易产生运动模糊+高ISO噪声’）进行综合推理，这是纯信号处理方法做不到的。

在实现上，我们设计了一个灵活的提示模板。模板包含：相机型号、ISO、快门、光圈、拍摄时间等字段。我们将这些填入预定义的prompt中，调用OpenAI API（或其他LLM服务）。为保证输出结构化，我们要求LLM以JSON格式回复，包含`noise_type`（枚举值）、`intensity`（低/中/高）、`description`（自然语言解释）三个字段。

数据流如下：输入是`DataCollector`产生的样本列表（每个含`image_path`和`exif`）；输出是增强后的样本列表，每个新增`llm_annotation`字段，内含噪声标签。这些标注将直接用于训练数据集的构建。

我们做了几个关键设计选择：第一，使用结构化输出（JSON schema）而非自由文本，确保下游能可靠解析；第二，加入重试机制——如果LLM返回无效JSON，自动重试最多3次；第三，缓存已标注结果，避免重复调用API浪费费用。这些选择平衡了准确性、成本和鲁棒性。

举个具体例子：输入样本的EXIF为{'iso': 3200, 'shutter_speed': 0.033, 'camera_model': 'iPhone 13'}。我们的prompt可能是：‘你是一位摄影噪声专家。请分析以下拍摄参数：相机=iPhone 13, ISO=3200, 快门=1/30秒。请以JSON格式输出最可能的噪声类型（选项：gaussian, poisson, salt_and_pepper, mixed）、强度（low/medium/high）及简要解释。’ LLM可能回复：{'noise_type': 'gaussian', 'intensity': 'high', 'description': 'iPhone在高ISO下主要产生高斯读出噪声'}。

边缘情况处理：如果EXIF缺失关键字段（如无ISO），我们在prompt中明确说明‘未知ISO’，让LLM基于其他信息推断；如果LLM持续返回无效响应，我们标记该样本为‘unlabeled’并记录日志，而不是中断整个流程。

最后，这个组件是连接真实世界数据与语义AI的关键桥梁。它的输出将直接决定后续扩散模型能否学会‘理解’噪声的语义。记住：**好的标注 = 好的监督信号 = 好的去噪效果**。


In [ ]:
import jsonimport timeimport loggingfrom typing import List, Dict, Optionalfrom openai import OpenAIlogger = logging.getLogger(__name__)class LLMAnnotator:    """    LLM噪声标注器：利用大语言模型根据EXIF元数据生成噪声语义标签。        功能：    - 将EXIF数据转化为结构化prompt    - 调用LLM API获取噪声类型、强度及描述    - 输出带语义标注的样本列表        参数:        api_key (str): OpenAI API密钥        model (str): 使用的LLM模型，默认gpt-4o-mini            返回:        增强后的样本列表，每个样本新增'llm_annotation'字段            示例:        annotator = LLMAnnotator("your-api-key")        annotated_samples = annotator.annotate(samples)    """        def __init__(self, api_key: str, model: str = "gpt-4o-mini"):        """        初始化LLM标注器。                Args:            api_key (str): OpenAI API密钥            model (str): LLM模型名称        """        if not api_key:            raise ValueError("API密钥不能为空")        self.client = OpenAI(api_key=api_key)        self.model = model        # 定义噪声类型枚举        self.noise_types = ["gaussian", "poisson", "salt_and_pepper", "mixed"]        def _build_prompt(self, exif_data: Dict[str, any]) -> str:        """        根据EXIF数据构建LLM提示。                Args:            exif_data (Dict): EXIF元数据字典                    Returns:            str: 构建好的提示文本        """        # 提取关键字段，处理缺失值        camera = exif_data.get('camera_model', '未知相机')        iso = exif_data.get('iso', '未知ISO')        shutter = exif_data.get('shutter_speed', '未知快门')        aperture = exif_data.get('aperture', '未知光圈')                # 格式化快门速度为分数（如0.033 -> "1/30"）        if isinstance(shutter, float) and shutter > 0:            # 简单近似：取倒数并四舍五入到常见分母            inv_shutter = 1.0 / shutter            rounded = round(inv_shutter)            shutter_str = f"1/{rounded}" if rounded > 1 else f"{shutter:.3f}s"        else:            shutter_str = str(shutter)                prompt = f"""你是一位专业的摄影噪声分析专家。请根据以下拍摄参数，分析图像中最可能存在的噪声类型、强度及原因。拍摄参数：- 相机型号: {camera}- ISO感光度: {iso}- 快门速度: {shutter_str}- 光圈值: {aperture}请严格按以下JSON格式输出，不要包含任何额外文本：{{  "noise_type": "gaussian|poisson|salt_and_pepper|mixed",  "intensity": "low|medium|high",  "description": "简要解释噪声来源（50字以内）"}}        """        return prompt.strip()        def _call_llm_with_retry(self, prompt: str, max_retries: int = 3) -> Optional[Dict]:        """        调用LLM并带重试机制，确保返回有效JSON。                Args:            prompt (str): LLM提示            max_retries (int): 最大重试次数                    Returns:            Optional[Dict]: 解析后的JSON响应，失败则返回None        """        for attempt in range(max_retries):            try:                response = self.client.chat.completions.create(                    model=self.model,                    messages=[                        {"role": "system", "content": "你是一个精确的JSON生成器。"},                        {"role": "user", "content": prompt}                    ],                    temperature=0.0,  # 降低随机性，提高确定性                    response_format={"type": "json_object"}  # 强制JSON输出                )                                content = response.choices[0].message.content                # 尝试解析JSON                parsed = json.loads(content)                                # 验证必要字段                if all(key in parsed for key in ['noise_type', 'intensity', 'description']):                    # 验证噪声类型是否在允许范围内                    if parsed['noise_type'] in self.noise_types:                        return parsed                    else:                        logger.warning(f"LLM返回无效噪声类型: {parsed['noise_type']}")                else:                    logger.warning(f"LLM返回缺少必要字段: {content}")                                except json.JSONDecodeError as e:                logger.warning(f"JSON解析失败 (尝试 {attempt+1}/{max_retries}): {str(e)}")            except Exception as e:                logger.warning(f"LLM调用异常 (尝试 {attempt+1}/{max_retries}): {str(e)}")                            # 重试前等待            if attempt < max_retries - 1:                time.sleep(1)                        return None        def annotate(self, samples: List[Dict]) -> List[Dict]:        """        为样本列表添加LLM噪声标注。                步骤：        1. 遍历每个样本        2. 构建prompt        3. 调用LLM获取标注        4. 将标注添加到样本中                Args:            samples (List[Dict]): 来自DataCollector的样本列表                    Returns:            List[Dict]: 增强后的样本列表（新增'llm_annotation'字段）        """        annotated_samples = []        total = len(samples)                for i, sample in enumerate(samples):            logger.info(f"正在标注样本 {i+1}/{total}: {sample['image_path']}")                        # 构建prompt            prompt = self._build_prompt(sample['exif'])                        # 调用LLM            annotation = self._call_llm_with_retry(prompt)                        if annotation is None:                logger.error(f"LLM标注失败，跳过样本: {sample['image_path']}")                # 添加空标注标记                sample['llm_annotation'] = None            else:                sample['llm_annotation'] = annotation                logger.debug(f"标注成功: {annotation}")                        annotated_samples.append(sample)                    success_count = sum(1 for s in annotated_samples if s['llm_annotation'] is not None)        logger.info(f"标注完成: {success_count}/{total} 成功")        return annotated_samples

#### 重要提示

- 强制JSON输出格式：通过OpenAI的`response_format={"type": "json_object"}`参数，显著提高结构化解析成功率，避免自由文本带来的解析错误。
- 成本与效率平衡：使用`gpt-4o-mini`而非`gpt-4`，在保持足够推理能力的同时大幅降低成本。对于大规模数据集，可考虑批量API或缓存机制进一步优化。
- 温度参数设置为0.0：确保LLM输出确定性结果，避免同一输入多次调用产生不同标注，保证数据集一致性。
- 隐私保护：EXIF中的GPS等敏感信息已在`DataCollector`中被过滤，此处仅使用摄影相关参数，符合数据安全规范。


### 1.3 噪声模拟器

**文件**: `src/noise_simulator.py`

**目的**: 在干净参考图像上模拟真实世界噪声（高斯、泊松、椒盐），用于数据增强和合成训练样本，同时支持根据LLM标注的噪声类型和强度动态调整噪声参数。

#### 详细说明

同学们，在上一步我们获得了真实图像的LLM语义标注（如‘高斯噪声，高强度’），但这里有个关键问题：**我们没有对应的干净参考图像（ground truth）**！真实拍摄的图像本身就是含噪的，无法直接用于监督训练（因为不知道‘干净版’长什么样）。为了解决这个根本矛盾，我们需要一个聪明的策略：**在已知干净图像上模拟符合LLM标注的噪声**。

`NoiseSimulator`就是执行这个策略的核心组件。它的输入有两个来源：一是公开的干净图像数据集（如DIV2K），二是上一步LLM生成的噪声标签。它会根据标签中的`noise_type`和`intensity`，在干净图像上施加相应类型和强度的噪声，从而生成成对的（干净图像, 合成含噪图像）训练样本。

为什么这样做合理？因为LLM的标注是基于真实拍摄条件的，而我们在干净图像上模拟的噪声参数（如高斯噪声的标准差）会根据强度等级动态调整（低=σ=10, 中=σ=25, 高=σ=50）。这样，合成的噪声分布就与真实世界统计特性对齐，既保证了训练监督信号的质量，又保留了语义一致性。

在实现上，我们支持三种基础噪声：
1. **高斯噪声**：添加均值为0、标准差σ的正态分布噪声。σ根据强度等级映射。
2. **泊松噪声**：模拟光子散粒噪声，噪声强度与像素值成正比。通过缩放因子控制强度。
3. **椒盐噪声**：随机将像素设为0或255，比例根据强度等级设定。

对于`mixed`类型，我们按概率组合多种噪声。数据流非常清晰：输入是干净图像路径列表 + LLM标注列表；输出是合成含噪图像列表，保存到`augmented_samples/`目录，并记录映射关系。

设计选择上，我们做了几个重要决策：第一，噪声参数与强度等级的映射表是可配置的（通过config.yaml），方便实验调整；第二，所有噪声操作在浮点域进行，最后裁剪到[0,255]并转回uint8，避免整数截断误差；第三，支持批量处理，利用NumPy向量化操作提升效率。

举个例子：假设LLM标注为{'noise_type': 'gaussian', 'intensity': 'high'}。我们的模拟器会加载一张干净图，添加σ=50的高斯噪声。如果标注是{'noise_type': 'poisson', 'intensity': 'medium'}，则应用缩放因子0.5的泊松噪声（即先除以0.5，取整，再乘回0.5）。

边缘情况处理：如果LLM标注缺失（`None`），我们跳过该样本；如果噪声类型未知，记录错误但继续处理。我们还加入了可视化调试选项——可保存少量样本的中间结果，方便检查噪声质量。

最后，这个组件是连接语义标注与实际训练数据的桥梁。它确保了：**合成噪声 = 真实噪声的统计近似 + LLM语义指导**。没有它，我们就无法获得大规模、高质量的配对训练数据。


In [ ]:
import osimport numpy as npfrom PIL import Imageimport yamlfrom typing import List

### 4 数据增强器

**文件**: `src/augmenter.py`

**目的**: 对原始含噪图像及其LLM生成的语义标签进行多样化增强，提升模型泛化能力，同时保持噪声语义一致性。

#### 详细说明

同学们，我们已经完成了真实噪声图像的收集（步骤1.1）、利用大语言模型为每张图像生成了噪声类型与强度的语义描述（步骤1.2），并构建了一个可模拟多种真实噪声模式的合成器（步骤1.3）。现在，我们面临一个关键挑战：**训练数据量有限，且真实拍摄场景存在高度多样性**。如果直接用原始数据训练去噪模型，很容易过拟合到特定相机型号、光照条件或拍摄角度。因此，我们需要引入**数据增强（Data Augmentation）**，但必须格外小心——普通的图像增强（如旋转、裁剪）可能会破坏噪声的空间结构或与LLM生成的语义标签不一致。

这就是本步骤的核心目标：设计一个**语义感知的数据增强器（Semantic-Aware Augmenter）**。它不仅要扩充样本数量，还要确保增强后的图像与其文本标签在语义上依然对齐。例如，如果我们对一张标注为“高ISO夜间手持拍摄导致的中等高斯-泊松混合噪声”的图像进行水平翻转，那么噪声的空间分布虽然改变了，但其物理成因和类型并未变化，因此标签仍然有效。但如果我们在增强过程中人为添加了椒盐噪声，而原始标签并未包含此类噪声，就会造成标签污染，误导后续模型训练。

我们的增强策略分为两类：**几何变换**（安全操作）和**噪声注入**（需谨慎控制）。几何变换包括随机裁剪、水平翻转、90度旋转等，这些操作不会改变噪声的统计特性，因此可以直接应用，并继承原始语义标签。而噪声注入则仅在特定条件下使用——例如，当原始图像噪声较弱时，我们可以基于步骤1.3中的`NoiseSimulator`，按照LLM预测的噪声类型和强度范围，**可控地**叠加额外噪声，从而生成“更强噪声”版本的样本，并相应更新其语义描述（如将“低强度”改为“中等强度”）。

在实现上，`Augmenter`类接收来自`LLMAnnotator`输出的标注数据（JSON格式，包含图像路径、原始标签、元数据），然后对每张图像执行一系列预设的增强策略。关键在于：**所有增强操作都必须记录其对语义标签的影响**。为此，我们设计了一个`_update_label_after_augmentation`方法，它会根据所执行的操作动态调整文本描述。例如，若进行了裁剪，我们会追加“局部区域”；若叠加了额外高斯噪声，则更新噪声强度等级。

数据流方面，输入是`annotated_dataset.json`中的条目列表，每个条目包含`image_path`、`noise_description`、`metadata`等字段。增强器遍历这些条目，对图像应用变换，生成新的图像文件（保存至`data/augmented_samples/`），并构建新的标注条目，最终输出一个扩展后的JSON数据集。这个新数据集将作为后续扩散模型训练的直接输入。

为什么选择这种设计？因为端到端的语义一致性是本研究的基石。如果增强破坏了“图像-文本”对齐，那么后续LLM引导的去噪过程将失去可靠的监督信号。我们放弃了全自动的强增强（如ColorJitter），因为色彩扰动会改变传感器噪声的感知特性；也避免了随机噪声叠加，除非有明确的语义依据。这是一种**受控增强（Controlled Augmentation）**哲学——增强是为了模拟真实世界的多样性，而非制造虚假样本。

举个具体例子：假设原始图像A的标签是“ISO 3200，轻微高斯噪声”。增强器可能对其进行水平翻转，生成图像A_flip，标签不变；也可能在确认其噪声强度低于阈值后，调用`NoiseSimulator.add_gaussian_noise`添加适量噪声，生成图像A_noisy，标签更新为“ISO 3200，中等高斯噪声”。这样，我们就用同一张原始图像生成了多个语义合理的变体。

边缘情况处理也很重要。例如，如果图像尺寸太小，无法进行有效裁剪，我们会跳过该操作；如果LLM标签缺失或格式错误，我们会记录警告但继续处理其他样本，确保流程鲁棒性。所有异常都会被日志记录，便于后期审计。

最后，这个组件与整个系统紧密耦合：它依赖于步骤1.2的标注结果和步骤1.3的噪声模拟能力，其输出将直接喂给后续的训练数据加载器。可以说，没有高质量的增强数据，再强大的扩散模型也无法学到泛化的去噪能力。因此，这一步虽看似“辅助”，实则是决定模型上限的关键环节。


In [ ]:
import osimport jsonimport cv2import numpy as npimport randomimport loggingfrom typing import List, Dict, Any, Optionalfrom pathlib import Path# 配置日志logging.basicConfig(level=logging.INFO)logger = logging.getLogger(__name__)class Augmenter:    """    语义感知的数据增强器：对含噪图像及其LLM生成的语义标签进行增强，    确保增强后的图像与更新后的文本标签保持语义一致性。    功能包括：    - 几何变换（安全操作，不改变噪声类型）    - 受控噪声注入（仅在原始噪声较弱时，按LLM预测类型叠加）    - 自动更新语义标签以反映增强操作    Args:        config (Dict[str, Any]): 配置字典，包含增强参数        noise_simulator: 已初始化的NoiseSimulator实例，用于可控噪声注入    Example:        >>> from noise_simulator import NoiseSimulator        >>> config = {'augment_factor': 3, 'min_noise_threshold': 15}        >>> simulator = NoiseSimulator()        >>> augmenter = Augmenter(config, simulator)        >>> augmented_data = augmenter.augment_dataset('data/annotated_dataset.json')    """    def __init__(self, config: Dict[str, Any], noise_simulator: Any):        self.config = config        self.noise_simulator = noise_simulator        self.augment_factor = config.get('augment_factor', 2)  # 每张图生成多少增强样本        self.min_noise_threshold = config.get('min_noise_threshold', 10)  # 噪声强度阈值（用于决定是否叠加）        self.output_dir = Path(config.get('output_dir', 'data/augmented_samples'))        self.output_dir.mkdir(parents=True, exist_ok=True)    def augment_dataset(self, annotated_json_path: str) -> List[Dict[str, Any]]:        """        对整个标注数据集进行增强，返回增强后的样本列表。        Args:            annotated_json_path (str): 原始标注JSON文件路径        Returns:            List[Dict]: 增强后的样本列表，每个元素包含新图像路径和更新后的标签        """        with open(annotated_json_path, 'r', encoding='utf-8') as f:            original_data = json.load(f)                augmented_samples = []                for idx, item in enumerate(original_data):            try:                image_path = item['image_path']                if not os.path.exists(image_path):                    logger.warning(f"图像不存在，跳过: {image_path}")                    continue                                image = cv2.imread(image_path)                if image is None:                    logger.warning(f"无法读取图像，跳过: {image_path}")                    continue                                # 为当前图像生成多个增强样本                for aug_idx in range(self.augment_factor):                    aug_image, aug_label = self._apply_augmentations(                        image.copy(),                         item['noise_description'],                         item.get('metadata', {})                    )                                        # 保存增强图像                    base_name = Path(image_path).stem                    new_filename = f"{base_name}_aug{aug_idx}.jpg"                    new_path = self.output_dir / new_filename                    cv2.imwrite(str(new_path), aug_image)                                        # 构建新样本条目                    new_item = {                        'original_image_path': image_path,                        'augmented_image_path': str(new_path),                        'noise_description': aug_label,                        'metadata': item.get('metadata', {}),                        'augmentation_applied': aug_idx  # 记录增强索引                    }                    augmented_samples.append(new_item)                                except Exception as e:                logger.error(f"处理图像 {image_path} 时出错: {str(e)}")                continue                logger.info(f"成功增强 {len(augmented_samples)} 个样本")        return augmented_samples    def _apply_augmentations(self, image: np.ndarray, label: str, metadata: Dict) -> tuple:        """        对单张图像应用一系列增强操作，并更新标签。        Args:            image (np.ndarray): 输入图像 (H, W, C)            label (str): 原始噪声语义描述            metadata (Dict): 图像元数据（如ISO、快门速度等）        Returns:            tuple: (增强后的图像, 更新后的标签)        """        current_label = label        h, w = image.shape[:2]                # === 步骤1: 几何变换（安全操作）===        # 随机水平翻转        if random.random() < 0.5:            image = cv2.flip(image, 1)  # 1表示水平翻转            # 标签无需改变，因为翻转不改变噪声物理特性                # 随机90度旋转        if random.random() < 0.3:            k = random.randint(1, 3)  # 旋转90, 180, 或270度            image = np.rot90(image, k)            # 同样，旋转不改变噪声类型，标签不变                # 随机裁剪（保留至少80%区域）        if min(h, w) > 200 and random.random() < 0.4:            crop_h = int(h * random.uniform(0.8, 1.0))            crop_w = int(w * random.uniform(0.8, 1.0))            y = random.randint(0, h - crop_h)            x = random.randint(0, w - crop_w)            image = image[y:y+crop_h, x:x+crop_w]            current_label = self._update_label_for_crop(current_label)                # === 步骤2: 受控噪声注入（需谨慎）===        # 仅当原始噪声较弱时才考虑叠加        if self._should_add_noise(metadata):            noise_type = self._infer_noise_type_from_label(current_label)            if noise_type == 'gaussian':                # 添加高斯噪声                std = random.uniform(10, 25)  # 控制强度                image = self.noise_simulator.add_gaussian_noise(image, std=std)                current_label = self._update_label_for_noise_strength(current_label, 'medium')            elif noise_type == 'poisson':                # 泊松噪声通常不直接叠加，跳过                pass            # 椒盐噪声较少见，此处暂不处理                return image, current_label    def _should_add_noise(self, metadata: Dict) -> bool:        """        根据元数据判断是否应叠加额外噪声。        例如，如果ISO较低（<800），说明原始噪声可能较弱。        """        iso = metadata.get('iso', 0)        # 如果ISO信息缺失，保守起见不叠加        if iso == 0:            return False        return iso < 800  # ISO低于800视为低噪声场景    def _infer_noise_type_from_label(self, label: str) -> str:        """        从LLM生成的文本标签中推断主要噪声类型。        这是一个简化版，实际可使用关键词匹配或小型分类器。        """        label_lower = label.lower()        if '高斯' in label_lower or 'gaussian' in label_lower:            return 'gaussian'        elif '泊松' in label_lower or 'poisson' in label_lower:            return 'poisson'        elif '椒盐' in label_lower or 'salt' in label_lower:            return 'salt_pepper'        else:            return 'unknown'    def _update_label_for_crop(self, label: str) -> str:        """        为裁剪操作更新标签，追加“局部区域”描述。        """        if '局部区域' not in label:            return label + "（局部区域）"        return label    def _update_label_for_noise_strength(self, label: str, new_strength: str) -> str:        """        更新噪声强度描述。        简单替换“轻微”、“低”等词为新强度。        """        # 移除旧的强度描述        for old in ['轻微', '低', 'low', 'mild']:            label = label.replace(old, '')        # 添加新强度        strength_map = {'medium': '中等'}        new_desc = strength_map.get(new_strength, new_strength)        return label + f"（{new_desc}强度）"

#### 重要提示

- 【语义一致性是核心】本增强器的关键创新在于动态更新文本标签以匹配图像变换。普通增强库（如Albumentations）只处理像素，而我们同时维护‘图像-文本’对齐，这是后续LLM引导去噪的前提。
- 【噪声注入需极度谨慎】我们仅在元数据表明原始噪声较弱时才叠加噪声，且仅限高斯类型。泊松噪声与信号相关，随意叠加会破坏物理真实性；椒盐噪声在真实相机中罕见，故暂不处理。
- 【几何变换的安全性】水平翻转、旋转、裁剪不会改变噪声的统计分布，因此标签无需大幅修改，只需追加‘局部区域’等上下文信息即可，这大大简化了标签更新逻辑。
- 【错误处理保障鲁棒性】代码中对图像读取失败、路径不存在等情况做了全面捕获，并记录日志而非中断流程，确保大规模数据处理时的稳定性。
- 【与NoiseSimulator的集成】增强器复用步骤1.3中的噪声模拟器，避免重复实现，体现了模块化设计思想。这种依赖关系通过构造函数注入，便于测试和替换。


### 5 主流程协调器

**文件**: `src/main.py`

**目的**: 协调整个数据准备流程，依次调用数据收集、LLM标注、噪声模拟和数据增强模块，生成最终训练数据集。

#### 详细说明

同学们，经过前四个步骤，我们已经分别实现了数据收集器（1.1）、LLM噪声标注器（1.2）、噪声模拟器（1.3）和数据增强器（步骤4）。现在，我们需要一个“指挥官”来把这些独立的模块**有机地串联起来**，形成一个端到端的自动化流水线。这就是`main.py`的角色——它不是功能模块，而是**流程编排器（Orchestrator）**，负责按正确顺序调用各个组件，并传递中间结果。

为什么需要这样一个主流程？因为在实际工程中，模块化开发虽好，但如果没有统一的入口点，团队协作和实验复现会变得极其困难。想象一下：你今天想用新采集的数据重新跑一遍标注和增强，明天想只测试增强效果……如果没有一个清晰的主脚本，你就要手动调用多个文件，极易出错。此外，配置管理（如路径、参数）也需要集中控制，避免硬编码散落在各处。

我们的主流程设计遵循**线性依赖链**：首先运行数据收集（输出`raw_images/`），然后用这些原始图像调用LLM标注器（输出`annotated_dataset.json`），接着用该JSON文件驱动数据增强器（内部可能调用噪声模拟器），最终生成完整的增强数据集。每一步的输出都是下一步的输入，形成清晰的数据血缘（Data Lineage）。

在实现上，`main.py`读取`configs/config.yaml`中的全局配置，然后依次实例化并调用各个组件。关键设计点在于**错误隔离与状态检查**：如果某一步失败（如LLM API超时），流程会停止并报错，而不是继续执行无效步骤。同时，我们会检查中间产物是否存在，避免重复计算——例如，如果`annotated_dataset.json`已存在且`force_reannotate=False`，就跳过LLM标注阶段。

数据流非常清晰：配置 → 数据收集 → LLM标注 → 数据增强 → 最终数据集。每一步都产生明确的输出文件或目录，这些路径都在配置文件中定义，便于修改。例如，你可以轻松切换不同的LLM服务（GPT-4 vs Claude）或调整增强倍数，只需改YAML文件，无需动代码。

我们选择YAML作为配置格式，因为它比JSON更易读写，支持注释，且能表达复杂嵌套结构。配置中包含了所有模块的参数：数据源路径、LLM API密钥、噪声模拟参数、增强策略等。这种**外部化配置**是生产级项目的标准实践。

举个运行示例：当你执行`python src/main.py --config configs/config.yaml`，程序会：
1. 从指定目录加载原始图像
2. 调用GPT-4为每张图生成噪声描述
3. 将标注结果存为JSON
4. 基于该JSON和噪声模拟器，生成10倍增强样本
5. 输出最终数据集路径和统计信息

边缘情况处理包括：空数据集检查、API密钥缺失提示、磁盘空间不足预警等。所有这些都通过日志输出，方便调试。

这个主流程看似简单，却是整个Package 1的“ glue code”（粘合代码）。它确保了从原始照片到训练数据的每一步都可追溯、可重复、可配置。没有它，我们的模块就像散落的珍珠；有了它，才能串成项链。

最后，这个脚本也为后续步骤（如模型训练）奠定了基础——训练脚本可以直接读取`augmented_samples/`和对应的JSON文件，无需关心数据是如何准备的。这种解耦设计让整个系统更灵活、更易维护。


In [ ]:
import argparseimport yamlimport osimport loggingfrom pathlib import Pathfrom data_collector import DataCollectorfrom llm_annotator import LLMAnnotatorfrom noise_simulator import NoiseSimulatorfrom augmenter import Augmenter# 配置日志logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')logger = logging.getLogger(__name__)def load_config(config_path: str) -> dict:    """    加载YAML配置文件。    Args:        config_path (str): 配置文件路径    Returns:        dict: 配置字典    """    with open(config_path, 'r', encoding='utf-8') as f:        return yaml.safe_load(f)def main():    """    主流程：协调数据准备全流程。    执行顺序：数据收集 → LLM标注 → 数据增强    """    parser = argparse.ArgumentParser(description='LLM引导的真实噪声图像数据准备流程')    parser.add_argument('--config', type=str, default='configs/config.yaml', help='配置文件路径')    parser.add_argument('--force-recollect', action='store_true', help='强制重新收集数据')    parser.add_argument('--force-reannotate', action='store_true', help='强制重新LLM标注')    args = parser.parse_args()    # 加载配置    config = load_config(args.config)    logger.info("配置加载成功")    # === 步骤1: 数据收集 ===    raw_dir = Path(config['data']['raw_images_dir'])    if not raw_dir.exists() or args.force_recollect:        logger.info("开始数据收集...")        collector = DataCollector(config['data_collector'])        collector.collect()        logger.info(f"数据收集完成，共 {len(list(raw_dir.glob('*.jpg')))} 张图像")    else:        logger.info(f"跳过数据收集，使用现有数据: {raw_dir}")    # 检查是否有原始图像    raw_images = list(raw_dir.glob('*.jpg'))    if not raw_images:        raise ValueError(f"原始图像目录为空: {raw_dir}")    # === 步骤2: LLM噪声标注 ===    annotated_path = Path(config['data']['annotated_json_path'])    if not annotated_path.exists() or args.force_reannotate:        logger.info("开始LLM噪声标注...")        annotator = LLMAnnotator(config['llm_annotator'])        annotated_data = annotator.annotate_directory(str(raw_dir))                # 保存标注结果        annotated_path.parent.mkdir(parents=True, exist_ok=True)        with open(annotated_path, 'w', encoding='utf-8') as f:            json.dump(annotated_data, f, ensure_ascii=False, indent=2)        logger.info(f"LLM标注完成，结果保存至: {annotated_path}")    else:        logger.info(f"跳过LLM标注，使用现有标注: {annotated_path}")    # === 步骤3: 数据增强 ===    # 初始化噪声模拟器（供增强器使用）    noise_sim = NoiseSimulator(config['noise_simulator'])        # 初始化增强器    augmenter_config = config['augmenter']    augmenter_config['output_dir'] = config['data']['augmented_dir']    augmenter = Augmenter(augmenter_config, noise_sim)        logger.info("开始数据增强...")    augmented_samples = augmenter.augment_dataset(str(annotated_path))        # 保存最终增强数据集    final_dataset_path = Path(config['data']['final_dataset_path'])    final_dataset_path.parent.mkdir(parents=True, exist_ok=True)    with open(final_dataset_path, 'w', encoding='utf-8') as f:        json.dump(augmented_samples, f, ensure_ascii=False, indent=2)        logger.info(f"数据增强完成！最终数据集包含 {len(augmented_samples)} 个样本")    logger.info(f"增强图像保存至: {config['data']['augmented_dir']}")    logger.info(f"最终标注文件: {final_dataset_path}")if __name__ == "__main__":    main()

#### 重要提示

- 【流程编排而非功能实现】main.py的核心价值在于协调各模块的执行顺序和数据传递，它本身不包含业务逻辑，这符合单一职责原则，使系统更易测试和维护。
- 【配置驱动一切】所有路径、参数都来自YAML配置，实现了代码与配置的分离。这意味着同一套代码可以轻松适配不同数据源或实验设置，极大提升复用性。
- 【智能跳过机制】通过检查中间文件是否存在及--force参数，避免了不必要的重复计算，节省时间和资源，这在处理大规模数据时尤为重要。
- 【错误前置检查】在进入下一步前，会验证上一步的输出（如检查原始图像是否存在），防止错误累积到后期才发现，提高调试效率。
- 【日志即文档】详细的日志输出不仅帮助调试，还自动记录了数据处理的全过程，为实验可复现性提供了保障。


### 6 配置文件

**文件**: `configs/config.yaml`

**目的**: 集中管理整个数据准备流程的参数和路径配置，实现代码与配置的分离，便于实验调整和部署。

#### 详细说明

同学们，在软件工程中有一条黄金法则：**永远不要把配置写死在代码里**。为什么？因为需求总是在变——今天你用GPT-4做标注，明天可能换成Claude；今天增强10倍，明天可能只需要5倍。如果这些参数都硬编码在Python文件中，每次调整都要改代码、测代码，效率极低且容易出错。

因此，我们引入了`config.yaml`这个**中央配置文件**。YAML（YAML Ain't Markup Language）是一种人类可读的数据序列化格式，比JSON更简洁，支持注释，非常适合做配置。在这个文件中，我们定义了整个Package 1所需的所有参数：数据路径、LLM API设置、噪声模拟参数、增强策略等。

让我们逐部分解析这个配置文件。首先是`data`部分，它定义了所有关键目录和文件路径：`raw_images_dir`是原始图像存放位置，`annotated_json_path`是LLM标注结果的输出路径，`augmented_dir`是增强图像的保存目录，`final_dataset_path`是最终数据集的JSON文件。这些路径都是相对项目根目录的，确保项目可移植。

接下来是`data_collector`配置。这里我们指定了数据源——可以是本地目录（`source: local`），也可以是云存储（未来可扩展）。`local_path`就是你的手机或相机照片所在位置。注意，我们还设置了`max_images`限制，防止意外加载过多图像导致内存溢出。

`llm_annotator`部分至关重要。它包含了调用大语言模型所需的一切：`model`指定使用哪个模型（如gpt-4-turbo），`api_key`是认证密钥（实际使用时应从环境变量读取，此处仅为示例），`prompt_template`定义了发送给LLM的提示词模板。这个模板非常关键——它告诉LLM如何根据图像元数据生成噪声描述。我们使用了占位符`{iso}`、`{shutter_speed}`等，这些会在运行时被实际值替换。

`noise_simulator`配置定义了各种噪声的默认参数。例如，高斯噪声的标准差范围、泊松噪声的缩放因子等。这些值基于真实相机传感器的典型特性设定，但你可以根据自己的数据调整。

最后，`augmenter`部分控制数据增强的行为：`augment_factor`决定每张图生成多少增强样本，`min_noise_threshold`用于判断是否叠加额外噪声（单位是像素标准差）。

这个配置文件如何被使用？在`main.py`中，我们通过`yaml.safe_load()`读取它，然后将对应的部分传递给各个组件。例如，`DataCollector`接收`config['data_collector']`，`LLMAnnotator`接收`config['llm_annotator']`。这种设计使得每个组件只关心自己的配置，降低了耦合度。

安全性方面，注意`api_key`不应明文写在配置文件中！在实际部署时，应通过环境变量或密钥管理服务注入。我们在示例中保留它只是为了教学清晰，但务必在真实项目中移除。

配置文件的另一个好处是**实验管理**。你可以创建多个YAML文件（如`config_low_noise.yaml`、`config_high_aug.yaml`），快速切换不同实验设置，而无需改动任何代码。这对于科研迭代至关重要。

总之，这个看似简单的YAML文件，实际上是整个数据准备流程的“控制面板”。它让我们的系统变得灵活、可配置、可复现——这正是专业级项目的标志。


In [ ]:
# Package 1: 基于大语言模型引导的真实噪声图像数据准备与语义标注框架# 全局配置文件# 数据路径配置data:  raw_images_dir: "data/raw_images/"          # 原始含噪图像目录  annotated_json_path: "data/annotated_dataset.json"  # LLM标注结果  augmented_dir: "data/augmented_samples/"    # 增强图像输出目录  final_dataset_path: "data/final_dataset.json"       # 最终数据集# 数据收集器配置data_collector:  source: "local"                             # 数据源类型: local | cloud (预留)  local_path: "/path/to/your/noisy/photos/"  # 本地原始图像路径 (需用户修改!)  max_images: 1000                            # 最大加载图像数量  extensions: [".jpg", ".jpeg", ".png"]      # 支持的图像格式# LLM噪声标注器配置llm_annotator:  model: "gpt-4-turbo"                        # 使用的LLM模型  api_key: "sk-your-api-key-here"             # OpenAI API密钥 (实际使用时应从环境变量读取!)  temperature: 0.3                            # 生成多样性控制 (值越低越确定)  max_tokens: 150                             # 最大生成长度  prompt_template: |    你是一位专业的摄影噪声分析专家。请根据以下图像元数据，生成一段简洁的中文描述，    说明该图像中最可能存在的噪声类型（高斯、泊松、椒盐或混合）及其强度（轻微、中等、严重）。    描述应包含噪声的物理成因（如高ISO、长曝光等）。        相机型号: {camera_model}    ISO感光度: {iso}    快门速度: {shutter_speed}    光圈: {aperture}    拍摄场景: {scene_description}        请直接输出描述，不要包含任何其他文字。# 噪声模拟器配置noise_simulator:  gaussian:    std_min: 5.0                              # 高斯噪声标准差最小值    std_max: 30.0                             # 高斯噪声标准差最大值  poisson:    scale_min: 0.5                            # 泊松噪声缩放因子最小值    scale_max: 2.0                            # 泊松噪声缩放因子最大值  salt_pepper:    prob_min: 0.01                            # 椒盐噪声概率最小值    prob_max: 0.05                            # 椒盐噪声概率最大值# 数据增强器配置augmenter:  augment_factor: 3                           # 每张原始图像生成的增强样本数  min_noise_threshold: 10                     # 噪声强度阈值 (低于此值才考虑叠加噪声)

#### 重要提示

- 【敏感信息保护】配置文件中的api_key仅为示例，实际项目中必须通过环境变量（如os.getenv('OPENAI_API_KEY')）或密钥管理服务注入，绝不能提交到代码仓库。
- 【路径可移植性】所有路径都使用相对路径（相对于项目根目录），确保项目在不同机器上都能正常运行，只需修改local_path指向你的数据位置。
- 【提示词模板设计】LLM的prompt_template经过精心设计，明确要求输出格式和内容，减少无关文本，这对后续自动解析标签至关重要。
- 【参数范围合理性】噪声模拟参数（如高斯std范围5-30）基于真实相机传感器噪声水平设定，过大或过小都会导致合成噪声不真实。
- 【实验友好性】通过复制此YAML文件并修改参数，可以轻松创建多个实验配置，无需改动代码，极大加速科研迭代。


### 7 使用文档

**文件**: `docs/usage.md`

**目的**: 提供清晰的使用指南，帮助用户快速上手本数据准备框架，包括环境安装、配置修改、运行命令和结果解读。

#### 详细说明

同学们，再好的代码，如果没有清晰的文档，也会让人望而却步。作为负责任的开发者和研究者，我们必须为使用者（包括未来的自己！）提供一份详尽的**使用手册**。这份`usage.md`文档位于`docs/`目录下，采用Markdown格式，既适合在GitHub上直接阅读，也方便转换为PDF或其他格式。

文档的结构遵循“由浅入深”的原则。首先，我们给出**一句话概述**，让用户立刻明白这个包是干什么的。接着是**先决条件**——你需要什么硬件、软件、账号才能运行它。例如，你需要Python 3.8+、OpenAI API密钥、以及一批真实的含噪图像。

然后是**分步指南**，这是文档的核心。我们将其拆解为四个清晰的步骤：1) 安装依赖；2) 准备数据；3) 配置参数；4) 运行主流程。每一步都配有具体的命令和截图（虽然此处是文本，但实际可附图），甚至包括常见错误的解决方案。例如，在“准备数据”部分，我们会提醒用户：“请将你的手机夜景照片放入`/path/to/your/noisy/photos/`，并确保它们包含EXIF元数据（大多数手机默认开启）”。

特别重要的是**配置说明**。我们会逐项解释`config.yaml`中每个参数的含义和推荐值。比如，对于`augment_factor`，我们会说明：“设为3表示每张原始图像生成3个增强样本，总计4倍数据量。如果你GPU内存有限，可设为1”。这种指导能极大降低新手的学习曲线。

我们还专门设置了**结果解读**章节。运行完成后，用户会得到一堆文件和目录，他们需要知道：`final_dataset.json`是什么结构？`augmented_samples/`里的图像如何使用？PSNR指标在哪里看？（虽然本包不计算PSNR，但我们会说明后续步骤会用到这些数据）。

为了应对现实问题，文档包含**故障排除（Troubleshooting）**部分。例如：“如果遇到‘Invalid API key’错误，请检查config.yaml中的api_key是否正确，并确认OpenAI账户余额充足”；“如果增强后的图像全是黑的，请检查原始图像是否损坏”。这些经验之谈能节省用户大量调试时间。

最后，我们提供**扩展建议**。比如：“如果你想支持更多噪声类型，可以修改noise_simulator.py中的add_custom_noise方法”；“若要使用本地LLM（如Llama 3），请替换llm_annotator.py中的API调用部分”。这鼓励用户在理解基础上进行创新。

这份文档不仅是说明书，更是**知识传承的载体**。它记录了设计决策、最佳实践和常见陷阱，让后来者站在我们的肩膀上前进。记住：优秀的开源项目，一半功劳在文档。


In [ ]:
# Package 1 使用指南：基于大语言模型引导的真实噪声图像数据准备## 概述本框架用于构建高质量、语义丰富的含噪图像数据集，专为LLM引导的扩散去噪模型设计。它能：- 从真实拍摄的低光/高ISO图像中收集数据- 利用GPT等大语言模型自动生成噪声类型与强度标签- 通过语义感知的数据增强扩充样本- 输出结构化数据集供后续训练使用## 先决条件- **Python 3.8+**- **OpenAI API 密钥**（或其他兼容LLM的API密钥）- **真实含噪图像**（建议：手机夜景模式、高ISO DSLR照片，需包含EXIF元数据）- **约10GB可用磁盘空间**（用于存储增强后的图像）## 快速开始### 1. 安装依赖```bashgit clone https://github.com/your-repo/package-01-llm-noise-annotation.gitcd package-01-llm-noise-annotationpip install -r requirements.txt```### 2. 准备原始数据- 将你的含噪图像（.jpg/.png）放入一个目录，例如 `~/my_noisy_photos/`- **重要**：确保图像包含EXIF元数据（ISO、快门速度等）。大多数手机和相机默认开启。### 3. 配置参数编辑 `configs/config.yaml`：```yaml# 修改这一行指向你的数据目录data_collector:  local_path: "/home/user/my_noisy_photos/"  # ←←← 在这里修改!# 替换为你的OpenAI API密钥 (强烈建议使用环境变量!)llm_annotator:  api_key: "sk-xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"```> **安全提示

---

## 📦 依赖安装

### 所需依赖



- **openai (>=1.0.0)**: 调用GPT系列大语言模型进行噪声语义标注


- **opencv-python (>=4.5.0)**: 图像加载、预处理和数据增强


- **numpy (>=1.21.0)**: 数值计算和噪声模拟


- **pyyaml (>=6.0)**: 解析配置文件


- **tqdm (>=4.60.0)**: 显示进度条，提升用户体验


In [ ]:
克隆本仓库：git clone https://github.com/your-repo/package-01-llm-noise-annotation.git
创建虚拟环境：python -m venv venv && source venv/bin/activate (Linux/Mac) 或 venv\Scripts\activate (Windows)
安装依赖：pip install -r requirements.txt
设置OpenAI API密钥：export OPENAI_API_KEY='your-api-key' (Linux/Mac) 或 set OPENAI_API_KEY=your-api-key (Windows)
准备原始图像：将真实含噪图像放入 data/raw_images/ 目录


---

## 🎮 使用教程


### 基础用法：自动标注单张图像

**场景**: 用户提供一张真实含噪图像和对应的元数据描述，系统调用LLM生成结构化噪声标签。


In [ ]:
from src.llm_annotator import LLMAnnotator# 初始化标注器annotator = LLMAnnotator(api_key="your-api-key")# 输入图像描述（模拟用户输入）description = "使用Canon EOS R6在ISO 3200、1/60秒快门下拍摄的室内人像，面部有明显彩色噪点，背景细节模糊。"# 生成标注label = annotator.annotate(description)print(label)

**预期输出**:

输出一个Python字典，例如：{'noise_type': ['gaussian'], 'intensity': 'high', 'source': 'high_iso_low_light'}。该结果将被保存到JSON文件中，供后续训练使用。


### 批量处理与数据增强

**场景**: 用户希望对整个raw_images目录中的图像进行批量标注，并对标注成功的样本进行安全的数据增强（如旋转、翻转），以扩充训练集。


In [ ]:
from src.main import process_dataset# 配置参数config = {    "input_dir": "data/raw_images",    "output_file": "data/annotated_dataset.json",    "augment": True,    "augment_dir": "data/augmented_samples",    "confidence_threshold": 0.8}# 执行批量处理process_dataset(config)

**预期输出**:

程序将遍历所有图像，为每张图生成描述（可通过EXIF自动提取或手动提供），调用LLM标注，过滤低置信度结果，并对高置信度样本进行旋转/翻转增强。最终生成annotated_dataset.json文件，包含图像路径、噪声标签、置信度等信息，同时增强图像保存在augmented_samples目录中。


---

## 📝 行动项

> [step_1] 数据准备与噪声标注 : 收集真实世界含噪声图像数据集（如相机拍摄、低光场景），并使用GPT类大语言模型辅助生成噪声类型（高斯、泊松、椒盐）及强度标签，结合数据增强策略扩充训练样本，提升模型泛化能力。



---

## 📚 参考文献

本包实现基于以下研究文献。在阅读理论基础和概念解释部分时，请注意文中引用的文献标记，如 [作者, 年份] 或 [序号]。



1. Jonathan Ho, Ajay Jain, P. Abbeel (2020). *Denoising Diffusion Probabilistic Models*. ArXiv
2. Prafulla Dhariwal, Alex Nichol (2021). *Diffusion Models Beat GANs on Image Synthesis*. ArXiv
3. Jiaming Song, Chenlin Meng, Stefano Ermon (2020). *Denoising Diffusion Implicit Models*. ArXiv
4. William S. Peebles, Saining Xie (2022). *Scalable Diffusion Models with Transformers*. 2023 IEEE/CVF International Conference on Computer Vision (ICCV)
5. Chitwan Saharia, William Chan, Saurabh Saxena et al. (2022). *Photorealistic Text-to-Image Diffusion Models with Deep Language Understanding*. ArXiv
6. Lvmin Zhang, Anyi Rao, Maneesh Agrawala (2023). *Adding Conditional Control to Text-to-Image Diffusion Models*. 2023 IEEE/CVF International Conference on Computer Vision (ICCV)
7. Nataniel Ruiz, Yuanzhen Li, Varun Jampani et al. (2022). *DreamBooth: Fine Tuning Text-to-Image Diffusion Models for Subject-Driven Generation*. 2023 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)
8. Kaiyang Zhou, Jingkang Yang, Chen Change Loy et al. (2022). *Conditional Prompt Learning for Vision-Language Models*. 2022 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)
9. Pengchuan Zhang, Xiujun Li, Xiaowei Hu et al. (2021). *VinVL: Revisiting Visual Representations in Vision-Language Models*. 2021 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)
10. Robin Rombach, A. Blattmann, Dominik Lorenz et al. (2021). *High-Resolution Image Synthesis with Latent Diffusion Models*. 2022 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)
11. Alex Nichol, Prafulla Dhariwal, A. Ramesh et al. (2021). *GLIDE: Towards Photorealistic Image Generation and Editing with Text-Guided Diffusion Models*. 
12. Boyuan Chen, Zhuo Xu, Sean Kirmani et al. (2024). *SpatialVLM: Endowing Vision-Language Models with Spatial Reasoning Capabilities*. 2024 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)
13. Xiaokang Peng, Yake Wei, Andong Deng et al. (2022). *Balanced Multimodal Learning via On-the-fly Gradient Modulation*. 2022 IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)
14. Dustin Podell, Zion English, Kyle Lacey et al. (2023). *SDXL: Improving Latent Diffusion Models for High-Resolution Image Synthesis*. ArXiv
15. Kaiyang Zhou, Jingkang Yang, Chen Change Loy et al. (2021). *Learning to Prompt for Vision-Language Models*. International Journal of Computer Vision
16. Wenliang Dai, Junnan Li, Dongxu Li et al. (2023). *InstructBLIP: Towards General-purpose Vision-Language Models with Instruction Tuning*. ArXiv
17. Deyao Zhu, Jun Chen, Xiaoqian Shen et al. (2023). *MiniGPT-4: Enhancing Vision-Language Understanding with Advanced Large Language Models*. ArXiv
18. Peng Gao, Shijie Geng, Renrui Zhang et al. (2021). *CLIP-Adapter: Better Vision-Language Models with Feature Adapters*. International Journal of Computer Vision
19. Yifan Li, Yifan Du, Kun Zhou et al. (2023). *Evaluating Object Hallucination in Large Vision-Language Models*. 
20. Anas Awadalla, Irena Gao, Josh Gardner et al. (2023). *OpenFlamingo: An Open-Source Framework for Training Large Autoregressive Vision-Language Models*. ArXiv
